- With path compression, rank becomes an estimate (compression flattens trees but ranks aren't decremented), so it's an upper bound, not the true height.

### DSU

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = [i for i in range(n)]
        self.rank = [0] * n # Track height(rank) of tree
        self.size = [1] * n # Track size of tree

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x, y):
        parent_x = self.find(x)
        parent_y = self.find(y)

        if parent_x == parent_y:
            return
        
        # Union by Rank
        if self.rank[parent_x] < self.rank[parent_y]:
            self.parent[parent_x] = parent_y
        elif self.rank[parent_x] > self.rank[parent_y]:
            self.parent[parent_y] = parent_x
        else:
            self.parent[parent_y] = parent_x
            self.rank[parent_x] += 1

        # Union by Size
        if self.size[parent_x] < self.size[parent_y]:
            self.parent[parent_x] = parent_y
            self.size[parent_y] += self.size[parent_x]
        else:
            self.parent[parent_y] = parent_x
            self.size[parent_x] += self.size[parent_y]

u = UnionFind(4)
u.union(3, 1)
u.union(1, 2)
print(u.find(2))

### Earliest all connected

In [ ]:
from typing import List

class UnionFind:
    def __init__(self, nodes):
        self.parent = {n: n for n in nodes}
        self.rank = {n: 0 for n in nodes}
        self.components = len(nodes)

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        self.components -= 1
        return True


def earliest_all_connected(logs: List[str], riders: List[str]):
    if not riders:
        return None
    if len(riders) == 1:
        # A single rider is trivially "all connected" at time 0 (or undefined).
        return 0

    uf = UnionFind(riders)
    rider_set = set(riders)

    for entry in logs:
        parts = entry.split()
        # parts = [timestamp, A, "shared-ride-with", B]
        if len(parts) < 4:
            continue
        ts, a, b = int(parts[0]), parts[1], parts[-1]

        # Ignore riders not in the known set (defensive)
        if a not in rider_set or b not in rider_set:
            continue

        uf.union(a, b)
        if uf.components == 1:
            return ts

    return None  # never fully connected

logs = [
    "167000000001 Alice shared-ride-with Bob",
    "167000000003 Charlie shared-ride-with Dan",
    "167000000008 Bob shared-ride-with Charlie",
    "167000000010 Alice shared-ride-with Eve",
    "167000000020 Bob shared-ride-with Dan",
]
riders = ["Alice", "Bob", "Charlie", "Dan", "Eve"]

print(earliest_all_connected(logs, riders))  # -> 167000000010

167000000010
